# Task 03 - End-to-End Machine Learning Project
## Telco Customer Churn Prediction
**Devixo Solutions - AI/ML Internship (Week 3)**

**Student:** Amtul Jamil Ansa  
**Dataset:** [Telco Customer Churn (Kaggle, IBM sample data)](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) - 7,043 customers, 21 columns

**Business problem:** A telecom company loses revenue every time a customer leaves ("churns"). If we can predict *which customers are likely to leave*, the company can contact them with offers before they go. This notebook builds a complete, reusable machine learning pipeline to make that prediction.

**Workflow**
1. Part 1 - Data preparation (load, clean, duplicates, feature engineering, feature selection, scaling)
2. Part 2 - Model development (Decision Tree, Random Forest, SVM, Gradient Boosting, XGBoost)
3. Part 3 - Model optimization (hyperparameter tuning, cross validation, feature importance)
4. Part 4 - Save the best model (Joblib and Pickle)
5. Part 5 - Documentation
6. Bonus - Streamlit prediction interface

## 0. Setup
We import the libraries and fix a random seed so results are reproducible.
`N_ITER` controls how many random hyperparameter combinations are tried per model (higher = better search but slower).

In [ ]:
!pip install -q kagglehub

import os, glob, json, time, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.stats import randint, uniform, loguniform

from sklearn.base import clone
from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV,
                                     RandomizedSearchCV, cross_val_score, cross_validate)
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.inspection import permutation_importance

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
N_ITER = 15        # random-search iterations per model
CV_FOLDS = 5
print("Setup complete. XGBoost available:", HAS_XGB)

### Reusable helper module
To make the project *reusable*, all data-cleaning and feature-engineering logic lives in one function, `prepare()`, saved to a file called `churn_utils.py`.
The same function is used for training here **and** for predictions in the Streamlit app, so raw customer data is always processed exactly the same way.

In [ ]:
%%writefile churn_utils.py
"""Shared preprocessing for the Telco churn project (used by the notebook and the Streamlit app)."""
import re
import numpy as np
import pandas as pd

YES_NO_COLS = ["Partner", "Dependents", "PhoneService", "PaperlessBilling"]
SERVICE_COLS = ["MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
                "TechSupport", "StreamingTV", "StreamingMovies"]
CATEGORICAL_COLS = ["InternetService", "Contract", "PaymentMethod", "tenure_group"]


def prepare(df, feature_columns=None):
    """Clean raw Telco customer data, engineer features and encode it numerically.

    If `feature_columns` is given, the output is re-indexed to exactly those columns
    (missing dummy columns are filled with 0) so a single customer can be scored.
    """
    d = df.copy()
    d = d.drop(columns=[c for c in ["customerID", "Churn"] if c in d.columns])

    # TotalCharges is stored as text; blanks belong to brand-new customers (tenure = 0)
    d["TotalCharges"] = pd.to_numeric(d["TotalCharges"].astype(str).str.strip(),
                                      errors="coerce").fillna(0)

    # "No internet service" / "No phone service" carry the same meaning as "No"
    for c in SERVICE_COLS:
        d[c] = d[c].replace({"No internet service": "No", "No phone service": "No"})

    # ---------- Feature engineering ----------
    d["tenure_group"] = pd.cut(d["tenure"], bins=[-1, 12, 24, 48, np.inf],
                               labels=["0-12m", "13-24m", "25-48m", "48m+"]).astype(str)
    d["avg_monthly_spend"] = d["TotalCharges"] / d["tenure"].clip(lower=1)
    d["charge_diff"] = d["MonthlyCharges"] - d["avg_monthly_spend"]
    d["num_services"] = ((d["InternetService"] != "No").astype(int)
                         + (d["PhoneService"] == "Yes").astype(int)
                         + sum((d[c] == "Yes").astype(int) for c in SERVICE_COLS if c != "MultipleLines"))
    d["is_autopay"] = d["PaymentMethod"].str.contains("automatic", case=False).astype(int)
    d["has_family"] = ((d["Partner"] == "Yes") | (d["Dependents"] == "Yes")).astype(int)

    # ---------- Encoding ----------
    d["gender"] = (d["gender"] == "Male").astype(int)
    for c in YES_NO_COLS + SERVICE_COLS:
        d[c] = (d[c] == "Yes").astype(int)
    d = pd.get_dummies(d, columns=CATEGORICAL_COLS)

    d.columns = [re.sub(r"[^0-9a-zA-Z_]+", "_", str(c)).strip("_") for c in d.columns]
    d = d.astype(float)
    if feature_columns is not None:
        d = d.reindex(columns=feature_columns, fill_value=0.0)
    return d

## Part 1 - Data Preparation
### 1.1 Import the dataset
The notebook downloads the dataset from Kaggle automatically. If that fails, it asks you to upload the CSV file manually
(download it from the Kaggle link above).

In [ ]:
CSV_NAME = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

if not os.path.exists(CSV_NAME):
    try:
        import kagglehub
        folder = kagglehub.dataset_download("blastchar/telco-customer-churn")
        csv_path = glob.glob(os.path.join(folder, "*.csv"))[0]
    except Exception as e:
        print("Automatic download failed:", e)
        from google.colab import files
        uploaded = files.upload()          # choose the Telco churn CSV
        csv_path = list(uploaded.keys())[0]
else:
    csv_path = CSV_NAME

import churn_utils                          # helper module written above
from churn_utils import prepare

df_raw = pd.read_csv(csv_path)
print("Rows, columns:", df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include="all").T

### 1.2 Handle missing values
`TotalCharges` looks numeric but is stored as text, which usually hides blank values. We convert it to numbers and see what is missing.

In [ ]:
df = df_raw.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].astype(str).str.strip(), errors="coerce")

missing = df.isna().sum()
print("Missing values per column (only columns with missing data):")
print(missing[missing > 0])

# Who are the customers with missing TotalCharges?
df[df["TotalCharges"].isna()][["customerID", "tenure", "MonthlyCharges", "TotalCharges"]].head(11)

All customers with a missing `TotalCharges` have `tenure = 0`: they just joined and have not been billed yet.
So the correct fill value is **0**, not the mean or median.

In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)
print("Remaining missing values:", int(df.isna().sum().sum()))

### 1.3 Remove duplicates
`customerID` is unique, so no row is identical *including* the ID. But the ID is not a useful feature, so we drop it and then check for rows that are identical in every other column.
Duplicates can leak between the training and test sets and make results look better than they are, so we remove them.

In [ ]:
print("Duplicates including customerID:", df.duplicated().sum())

df = df.drop(columns="customerID")
n_before = len(df)
print("Duplicates ignoring customerID :", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)
print(f"Rows before: {n_before}  |  after: {len(df)}  |  removed: {n_before - len(df)}")

### 1.4 Quick exploratory analysis
A few plots to understand the problem before modelling.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Target balance
counts = df["Churn"].value_counts()
axes[0, 0].bar(counts.index, counts.values, color=["#4C9F70", "#D9534F"])
axes[0, 0].set_title("Churn distribution (class balance)")
for i, v in enumerate(counts.values):
    axes[0, 0].text(i, v + 30, f"{v} ({v / len(df):.1%})", ha="center")

# Churn rate by contract
rate = df.groupby("Contract")["Churn"].apply(lambda s: (s == "Yes").mean()).sort_values()
axes[0, 1].bar(rate.index, rate.values, color="#5B8DB8")
axes[0, 1].set_title("Churn rate by contract type")
axes[0, 1].set_ylabel("Churn rate")

# Tenure
sns.histplot(data=df, x="tenure", hue="Churn", bins=30, multiple="stack",
             palette=["#4C9F70", "#D9534F"], ax=axes[1, 0])
axes[1, 0].set_title("Tenure (months) by churn")

# Monthly charges
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", palette=["#4C9F70", "#D9534F"], ax=axes[1, 1])
axes[1, 1].set_title("Monthly charges by churn")

plt.tight_layout()
plt.show()

**What we see:** about a quarter of customers churn (so the classes are *imbalanced*), month-to-month contracts churn far more than long contracts,
churners tend to have short tenure, and they pay higher monthly charges. These observations guide the features we create next.

### 1.5 Feature engineering
We create new columns that capture customer behaviour better than the raw columns:

| New feature | Meaning |
|---|---|
| `tenure_group` | Customer age bucket: 0-12, 13-24, 25-48, 48+ months |
| `avg_monthly_spend` | TotalCharges / tenure - what the customer has paid per month on average |
| `charge_diff` | Current MonthlyCharges minus average spend (positive = bill has risen) |
| `num_services` | How many services the customer subscribes to |
| `is_autopay` | 1 if paying by automatic bank transfer or credit card |
| `has_family` | 1 if the customer has a partner or dependents |

Yes/No columns are converted to 1/0 and multi-category columns are one-hot encoded. All of this happens inside `prepare()`.

In [ ]:
y = (df["Churn"] == "Yes").astype(int)
X_raw = df.drop(columns="Churn")

X = prepare(X_raw)
print("Shape after cleaning, feature engineering and encoding:", X.shape)

new_feats = ["avg_monthly_spend", "charge_diff", "num_services", "is_autopay", "has_family"]
print("\nAverage of engineered features, split by churn (0 = stayed, 1 = churned):")
display(X[new_feats].groupby(y).mean().round(2))
X.head()

### 1.6 Train / test split
We hold out 20% of the data as a **test set** that is never used for training, tuning or feature selection. The split is *stratified* so both sets keep the same churn percentage.
Everything that "learns" from data (feature selection, scaling, tuning) is done on the training set only, to avoid data leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Churn rate - train: %.3f | test: %.3f" % (y_train.mean(), y_test.mean()))

### 1.7 Feature selection
Two steps, both using the training data only:
1. **Remove highly correlated features** (|correlation| > 0.90). Two features that say almost the same thing add noise, not information.
2. **Mutual information filter.** Mutual information measures how much knowing a feature reduces uncertainty about churn. Features with almost no information are dropped.

In [ ]:
# Step 1 - correlated features
corr = X_train.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_corr = [c for c in upper.columns if (upper[c] > 0.90).any()]
print("Dropped for high correlation:", drop_corr if drop_corr else "none")

plt.figure(figsize=(12, 9))
sns.heatmap(X_train.corr(), cmap="coolwarm", center=0, xticklabels=False, yticklabels=False)
plt.title("Correlation matrix of all engineered features (training set)")
plt.show()

In [ ]:
# Step 2 - mutual information
cols_after_corr = [c for c in X_train.columns if c not in drop_corr]
is_discrete = [X_train[c].nunique() <= 2 for c in cols_after_corr]

mi = pd.Series(
    mutual_info_classif(X_train[cols_after_corr], y_train,
                        discrete_features=is_discrete, random_state=RANDOM_STATE),
    index=cols_after_corr).sort_values(ascending=False)

MI_THRESHOLD = 0.005
selected_features = mi[mi >= MI_THRESHOLD].index.tolist()
dropped_mi = mi[mi < MI_THRESHOLD].index.tolist()

print(f"Features available: {X.shape[1]} | after correlation filter: {len(cols_after_corr)} "
      f"| selected: {len(selected_features)}")
print("Dropped for low mutual information:", dropped_mi if dropped_mi else "none")

plt.figure(figsize=(9, 9))
mi.sort_values().plot(kind="barh", color=["#D9534F" if v < MI_THRESHOLD else "#5B8DB8" for v in mi.sort_values()])
plt.axvline(MI_THRESHOLD, color="black", linestyle="--", label=f"threshold = {MI_THRESHOLD}")
plt.title("Mutual information with churn (red = dropped)")
plt.legend()
plt.tight_layout()
plt.show()

### 1.8 Data scaling
Features such as `TotalCharges` (up to ~8,000) and `tenure` (0-72) are on very different scales than 0/1 columns.
Distance-based models like SVM need all features on a comparable scale. We use **StandardScaler** (mean 0, standard deviation 1) on the numeric columns only.

The scaler is placed **inside the model pipeline**, so during cross validation it is fitted only on each training fold. This is the correct way to avoid leakage.

In [ ]:
numeric_candidates = ["tenure", "MonthlyCharges", "TotalCharges",
                      "avg_monthly_spend", "charge_diff", "num_services"]
numeric_cols = [c for c in numeric_candidates if c in selected_features]

preprocessor = ColumnTransformer(
    [("scale", StandardScaler(), numeric_cols)],
    remainder="passthrough")

Xtr = X_train[selected_features]
Xte = X_test[selected_features]

# Demonstration: numeric columns before and after scaling
scaled_demo = pd.DataFrame(StandardScaler().fit_transform(Xtr[numeric_cols]), columns=numeric_cols)
print("BEFORE scaling:")
display(Xtr[numeric_cols].describe().loc[["mean", "std", "min", "max"]].round(2))
print("AFTER scaling:")
display(scaled_demo.describe().loc[["mean", "std", "min", "max"]].round(2))

## Part 2 - Model Development
We train five different models, each inside the same pipeline (scaling then model):

| Model | Idea in one line |
|---|---|
| Decision Tree | A flowchart of yes/no questions about the customer |
| Random Forest | Many decision trees voting together |
| Support Vector Machine | Finds the best boundary separating churners from stayers |
| Gradient Boosting | Trees built one after another, each fixing the previous one's mistakes |
| XGBoost (optional) | A faster, regularised version of gradient boosting |

Because only ~27% of customers churn, we use `class_weight="balanced"` where available so the models do not simply predict "stays" for everyone.

**Metrics:** Accuracy alone is misleading on imbalanced data, so we also report Precision, Recall, F1 and ROC-AUC.
For churn, **Recall** matters (how many real churners we catch) and **ROC-AUC** measures how well the model ranks risky customers above safe ones.

In [ ]:
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos

models = {
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                            random_state=RANDOM_STATE, n_jobs=-1),
    "SVM": SVC(kernel="rbf", class_weight="balanced", random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}
if HAS_XGB:
    models["XGBoost"] = XGBClassifier(eval_metric="logloss", scale_pos_weight=scale_pos_weight,
                                      random_state=RANDOM_STATE, n_jobs=-1)

def make_pipe(model):
    return Pipeline([("prep", clone(preprocessor)), ("model", clone(model))])

def get_scores(pipe, X_):
    """Probability of churn if available, otherwise the SVM decision score."""
    if hasattr(pipe, "predict_proba"):
        return pipe.predict_proba(X_)[:, 1]
    return pipe.decision_function(X_)

def evaluate_on(pipe, X_, y_):
    pred = pipe.predict(X_)
    return {"Accuracy": accuracy_score(y_, pred),
            "Precision": precision_score(y_, pred),
            "Recall": recall_score(y_, pred),
            "F1": f1_score(y_, pred),
            "ROC-AUC": roc_auc_score(y_, get_scores(pipe, X_))}

cv5 = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

baseline_rows, baseline_pipes = [], {}
for name, model in models.items():
    t0 = time.time()
    pipe = make_pipe(model).fit(Xtr, y_train)
    row = evaluate_on(pipe, Xte, y_test)
    row["CV ROC-AUC (train)"] = cross_val_score(pipe, Xtr, y_train, cv=cv5,
                                               scoring="roc_auc", n_jobs=-1).mean()
    row["Model"] = name
    baseline_rows.append(row)
    baseline_pipes[name] = pipe
    print(f"{name:<18} trained in {time.time() - t0:5.1f}s")

baseline_df = pd.DataFrame(baseline_rows).set_index("Model").sort_values("ROC-AUC", ascending=False)
print("\nBASELINE RESULTS (default settings, evaluated on the held-out test set)")
baseline_df.round(4)

In [ ]:
ax = baseline_df[["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]].plot(
    kind="bar", figsize=(12, 5), width=0.8)
ax.set_title("Baseline model comparison (test set)")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc="lower right", ncol=5)
plt.tight_layout()
plt.show()

## Part 3 - Model Optimization
### 3.1 Hyperparameter tuning
Every model has settings (hyperparameters) that are not learned from data, such as tree depth. We search for the best ones using **cross-validated search on the training set**:

- **GridSearchCV** for the Decision Tree - its search space is small, so we try every combination.
- **RandomizedSearchCV** for the other models - their search spaces are large, so we sample `N_ITER` random combinations.

Each candidate is scored with 5-fold cross validation using ROC-AUC. The test set is not touched.

In [ ]:
dt_grid = {
    "model__max_depth": [3, 5, 7, 10, None],
    "model__min_samples_leaf": [1, 5, 10, 20],
    "model__criterion": ["gini", "entropy"],
}
rf_dist = {
    "model__n_estimators": randint(100, 400),
    "model__max_depth": [None, 5, 8, 12, 16],
    "model__min_samples_leaf": randint(1, 15),
    "model__min_samples_split": randint(2, 20),
    "model__max_features": ["sqrt", "log2", 0.5],
}
svm_dist = {
    "model__C": loguniform(0.1, 50),
    "model__gamma": loguniform(1e-4, 1e-1),
}
gb_dist = {
    "model__n_estimators": randint(100, 400),
    "model__learning_rate": loguniform(0.01, 0.3),
    "model__max_depth": randint(2, 5),
    "model__subsample": uniform(0.6, 0.4),
    "model__min_samples_leaf": randint(1, 20),
}
xgb_dist = {
    "model__n_estimators": randint(100, 500),
    "model__learning_rate": loguniform(0.01, 0.3),
    "model__max_depth": randint(2, 7),
    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),
}

def rand_search(name, dist):
    return RandomizedSearchCV(make_pipe(models[name]), dist, n_iter=N_ITER, scoring="roc_auc",
                              cv=cv5, n_jobs=-1, random_state=RANDOM_STATE, refit=True)

searches = {
    "Decision Tree": GridSearchCV(make_pipe(models["Decision Tree"]), dt_grid,
                                  scoring="roc_auc", cv=cv5, n_jobs=-1, refit=True),
    "Random Forest": rand_search("Random Forest", rf_dist),
    "SVM": rand_search("SVM", svm_dist),
    "Gradient Boosting": rand_search("Gradient Boosting", gb_dist),
}
if HAS_XGB:
    searches["XGBoost"] = rand_search("XGBoost", xgb_dist)

tuned = {}
for name, search in searches.items():
    t0 = time.time()
    search.fit(Xtr, y_train)
    tuned[name] = search.best_estimator_
    print(f"{name:<18} done in {time.time() - t0:6.1f}s | best CV ROC-AUC = {search.best_score_:.4f}")

In [ ]:
tuning_summary = pd.DataFrame({
    "Baseline CV ROC-AUC": baseline_df["CV ROC-AUC (train)"],
    "Tuned CV ROC-AUC": pd.Series({n: s.best_score_ for n, s in searches.items()}),
})
tuning_summary["Improvement"] = tuning_summary["Tuned CV ROC-AUC"] - tuning_summary["Baseline CV ROC-AUC"]
display(tuning_summary.sort_values("Tuned CV ROC-AUC", ascending=False).round(4))

print("Best hyperparameters found:")
for name, s in searches.items():
    clean = {k.replace("model__", ""): (round(v, 4) if isinstance(v, float) else v)
             for k, v in s.best_params_.items()}
    print(f"  {name}: {clean}")

### 3.2 Cross validation
Cross validation splits the training data into 5 folds. The model is trained on 4 folds and tested on the 5th, rotating five times.
The **mean** tells us typical performance and the **standard deviation** tells us how stable the model is.
Here we use a fresh set of folds (different random seed) to evaluate the tuned models.

In [ ]:
cv_eval = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=7)
scoring = {"accuracy": "accuracy", "precision": "precision", "recall": "recall",
           "f1": "f1", "roc_auc": "roc_auc"}

cv_rows, fold_auc = [], {}
for name, est in tuned.items():
    res = cross_validate(est, Xtr, y_train, cv=cv_eval, scoring=scoring, n_jobs=-1)
    fold_auc[name] = res["test_roc_auc"]
    row = {"Model": name}
    for m in scoring:
        row[f"{m} mean"] = res[f"test_{m}"].mean()
        row[f"{m} std"] = res[f"test_{m}"].std()
    cv_rows.append(row)

cv_table = pd.DataFrame(cv_rows).set_index("Model").sort_values("roc_auc mean", ascending=False)
print("5-FOLD CROSS VALIDATION - TUNED MODELS (training data)")
display(cv_table.round(4))

plt.figure(figsize=(9, 5))
plt.boxplot(list(fold_auc.values()))
plt.xticks(range(1, len(fold_auc) + 1), list(fold_auc.keys()))
plt.title("ROC-AUC across the 5 cross-validation folds (tuned models)")
plt.ylabel("ROC-AUC")
plt.show()

### 3.3 Final evaluation on the unseen test set
Now the tuned models face the test set for the first time. This is the honest estimate of how they would perform on new customers.

In [ ]:
final_rows = []
for name, est in tuned.items():
    r = evaluate_on(est, Xte, y_test)
    r["Model"] = name
    final_rows.append(r)

final_df = pd.DataFrame(final_rows).set_index("Model").sort_values("ROC-AUC", ascending=False)
print("TUNED MODELS - TEST SET RESULTS")
display(final_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for name, est in tuned.items():
    RocCurveDisplay.from_estimator(est, Xte, y_test, name=name, ax=axes[0])
axes[0].plot([0, 1], [0, 1], "k--", label="Random guess")
axes[0].set_title("ROC curves - tuned models (test set)")
axes[0].legend(loc="lower right")

final_df[["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]].plot(kind="bar", ax=axes[1], width=0.8)
axes[1].set_title("Tuned model comparison (test set)")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
n = len(tuned)
fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 4))
for ax, (name, est) in zip(np.atleast_1d(axes), tuned.items()):
    ConfusionMatrixDisplay.from_estimator(est, Xte, y_test, display_labels=["Stayed", "Churned"],
                                          cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(name)
plt.tight_layout()
plt.show()

### Choosing the best model
To pick the winner fairly we use the **cross-validation ROC-AUC on the training data**, not the test score. Choosing by test score would slowly "leak" the test set into our decisions.
The test results above are then the unbiased report card of the chosen model.

In [ ]:
best_name = cv_table["roc_auc mean"].idxmax()
best_pipe = tuned[best_name]

print(f"BEST MODEL: {best_name}")
print(f"  CV ROC-AUC (train): {cv_table.loc[best_name, 'roc_auc mean']:.4f} +/- {cv_table.loc[best_name, 'roc_auc std']:.4f}")
print(f"  Test ROC-AUC      : {final_df.loc[best_name, 'ROC-AUC']:.4f}\n")
print(classification_report(y_test, best_pipe.predict(Xte), target_names=["Stayed", "Churned"]))

### 3.4 Feature importance analysis
Which customer characteristics drive churn predictions? We look at two views:
1. **Built-in importance** of a tree-based model (how much each feature reduces impurity across the trees).
2. **Permutation importance** (model-agnostic): shuffle one feature at a time on the test set and measure how much the ROC-AUC drops. A big drop means the model relies on that feature.

In [ ]:
def pipe_feature_names(pipe):
    return [n.split("__", 1)[1] for n in pipe.named_steps["prep"].get_feature_names_out()]

tree_name = best_name if hasattr(best_pipe.named_steps["model"], "feature_importances_") else "Random Forest"
tree_pipe = tuned[tree_name]
imp = pd.Series(tree_pipe.named_steps["model"].feature_importances_,
                index=pipe_feature_names(tree_pipe)).sort_values(ascending=False)

perm = permutation_importance(best_pipe, Xte, y_test, scoring="roc_auc",
                              n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
perm_s = pd.Series(perm.importances_mean, index=Xte.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
imp.head(15)[::-1].plot(kind="barh", ax=axes[0], color="#5B8DB8")
axes[0].set_title(f"Top 15 built-in feature importances ({tree_name})")
perm_s.head(15)[::-1].plot(kind="barh", ax=axes[1], color="#D9534F")
axes[1].set_title(f"Top 15 permutation importances ({best_name}, test set)")
axes[1].set_xlabel("Drop in ROC-AUC when feature is shuffled")
plt.tight_layout()
plt.show()

print("Top 5 drivers (permutation):", ", ".join(perm_s.head(5).index))

**Reading the chart:** features at the top are what the model leans on most. Typically contract type, tenure and charges dominate, which matches the patterns in our exploratory analysis.
These are also actionable business levers (for example, offering long-term contracts to month-to-month customers).

## Part 4 - Save the Best Model
We save the **entire pipeline** (scaler + tuned model) in two formats:
- **Joblib** - the standard way to save scikit-learn models; efficient for models containing large NumPy arrays.
- **Pickle** - Python's built-in serialization format; works for any Python object.

We also save a small JSON file with the list of selected features, so the app knows exactly which inputs the model expects.

In [ ]:
# The Streamlit app needs probabilities. SVM only provides them if probability=True,
# so in that case we refit the already-tuned pipeline with probability enabled.
if not hasattr(best_pipe, "predict_proba"):
    best_pipe = clone(best_pipe).set_params(model__probability=True).fit(Xtr, y_train)

os.makedirs("models", exist_ok=True)
joblib.dump(best_pipe, "models/best_churn_model.joblib")
with open("models/best_churn_model.pkl", "wb") as f:
    pickle.dump(best_pipe, f)

metadata = {
    "best_model": best_name,
    "selected_features": selected_features,
    "all_feature_columns": X.columns.tolist(),
    "numeric_features_scaled": numeric_cols,
    "test_metrics": {k: round(float(v), 4) for k, v in final_df.loc[best_name].items()},
    "random_state": RANDOM_STATE,
}
with open("models/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

for p in sorted(glob.glob("models/*")):
    print(f"{p:<38} {os.path.getsize(p) / 1024:8.1f} KB")

In [ ]:
# Verify that both saved files reload and give identical predictions
m_joblib = joblib.load("models/best_churn_model.joblib")
with open("models/best_churn_model.pkl", "rb") as f:
    m_pickle = pickle.load(f)

p0 = best_pipe.predict_proba(Xte)[:, 1]
print("Joblib reload identical:", np.allclose(p0, m_joblib.predict_proba(Xte)[:, 1]))
print("Pickle reload identical:", np.allclose(p0, m_pickle.predict_proba(Xte)[:, 1]))

### Using the saved model on a brand-new customer
This is the whole point of an end-to-end project: raw customer details go in, a churn probability comes out.

In [ ]:
def predict_churn(customer: dict, threshold=0.5):
    """Score one raw customer record with the saved model."""
    model = joblib.load("models/best_churn_model.joblib")
    meta = json.load(open("models/model_metadata.json"))
    feats = prepare(pd.DataFrame([customer]), feature_columns=meta["all_feature_columns"])
    prob = float(model.predict_proba(feats[meta["selected_features"]])[0, 1])
    return prob, ("Likely to CHURN" if prob >= threshold else "Likely to STAY")

risky_customer = {
    "gender": "Female", "SeniorCitizen": 0, "Partner": "No", "Dependents": "No", "tenure": 2,
    "PhoneService": "Yes", "MultipleLines": "No", "InternetService": "Fiber optic",
    "OnlineSecurity": "No", "OnlineBackup": "No", "DeviceProtection": "No", "TechSupport": "No",
    "StreamingTV": "Yes", "StreamingMovies": "Yes", "Contract": "Month-to-month",
    "PaperlessBilling": "Yes", "PaymentMethod": "Electronic check",
    "MonthlyCharges": 95.5, "TotalCharges": 191.0}

loyal_customer = {
    "gender": "Male", "SeniorCitizen": 0, "Partner": "Yes", "Dependents": "Yes", "tenure": 68,
    "PhoneService": "Yes", "MultipleLines": "Yes", "InternetService": "DSL",
    "OnlineSecurity": "Yes", "OnlineBackup": "Yes", "DeviceProtection": "Yes", "TechSupport": "Yes",
    "StreamingTV": "No", "StreamingMovies": "No", "Contract": "Two year",
    "PaperlessBilling": "No", "PaymentMethod": "Bank transfer (automatic)",
    "MonthlyCharges": 64.9, "TotalCharges": 4400.0}

for label, cust in [("New, month-to-month, fiber, no add-ons", risky_customer),
                    ("Long-tenure, 2-year contract, many add-ons", loyal_customer)]:
    prob, verdict = predict_churn(cust)
    print(f"{label:<45} -> churn probability {prob:6.1%}  =>  {verdict}")

## Part 5 - Project Documentation

### Dataset
- **Source:** Telco Customer Churn, Kaggle (`blastchar/telco-customer-churn`), IBM sample data.
- **Size:** 7,043 customers, 21 columns (demographics, services subscribed, account and billing information, and the target `Churn`).
- **Target:** `Churn` (Yes/No) - whether the customer left within the last month. About 27% of customers churned (imbalanced classes).

### Workflow
1. Loaded the data and found that `TotalCharges` was stored as text with blanks for brand-new customers (tenure 0), which were filled with 0.
2. Dropped the ID column and removed duplicate records.
3. Engineered six behavioural features and encoded all categorical columns.
4. Split 80/20 (stratified) into training and test sets.
5. Selected features on the training set using a correlation filter and mutual information.
6. Scaled numeric features with StandardScaler inside a scikit-learn Pipeline (prevents leakage).
7. Trained Decision Tree, Random Forest, SVM, Gradient Boosting and XGBoost.
8. Tuned each with GridSearchCV / RandomizedSearchCV (5-fold CV, ROC-AUC) and validated with a second cross-validation run.
9. Selected the best model by cross-validated ROC-AUC, then evaluated it once on the test set.
10. Analysed feature importance, saved the model with Joblib and Pickle, and built a Streamlit app.

### Model selection
Models were compared on ROC-AUC (threshold-independent ranking quality) alongside Recall and F1, because accuracy alone is misleading with imbalanced classes and the business cost of *missing* a churner is higher than the cost of a wasted retention offer.
The final choice was made on cross-validation results from the training data so the test set stayed an unbiased final check.

### Limitations
- Data comes from a single snapshot of a fictional/sample telecom; real customer behaviour changes over time.
- Only about 7,000 customers and no time dimension (we cannot see *when* behaviour changed before leaving).
- No information on customer service interactions, network quality or competitor offers, which strongly influence real churn.
- The default 0.5 decision threshold is not necessarily optimal; the right threshold depends on the cost of a retention offer versus the value of a customer.
- Correlation is not causation: feature importance shows what predicts churn, not what causes it.

### Future improvements
- Tune the decision threshold using business costs (retention offer cost vs. customer lifetime value).
- Try SMOTE or other resampling, stacking/ensembling, and a larger tuning budget (Bayesian optimization, e.g. Optuna).
- Add explainability with SHAP to explain each individual prediction.
- Add time-based data and customer service features; validate on a time-based split.
- Deploy the app online (Streamlit Community Cloud) and monitor the model for drift.

In [ ]:
# Final results summary - convenient for the report and screenshots
print("=" * 60)
print("PROJECT SUMMARY")
print("=" * 60)
print(f"Dataset              : Telco Customer Churn ({df_raw.shape[0]} raw rows, {df_raw.shape[1]} columns)")
print(f"Rows after cleaning  : {len(df)}  (churn rate {y.mean():.1%})")
print(f"Features             : {X.shape[1]} engineered -> {len(selected_features)} selected")
print(f"Train / test         : {len(X_train)} / {len(X_test)}")
print(f"Models trained       : {', '.join(models)}")
print(f"Best model           : {best_name}")
for k, v in final_df.loc[best_name].items():
    print(f"   Test {k:<10}: {v:.4f}")
print("=" * 60)

## Bonus - Streamlit Prediction Interface
The next cell writes `app.py`, a small web app where a user enters customer details and gets the predicted churn probability.
It reuses `churn_utils.prepare()` and the saved model, so predictions match the notebook exactly.

**To run it on your own computer** (not inside Colab):
1. Put `app.py`, `churn_utils.py` and the `models/` folder in one directory (the download cell at the end of this notebook packs them for you).
2. Open a terminal in that directory and run:
```
pip install streamlit scikit-learn pandas numpy joblib xgboost
streamlit run app.py
```
3. Your browser opens at `http://localhost:8501`.

In [ ]:
%%writefile app.py
import json
import joblib
import pandas as pd
import streamlit as st
from churn_utils import prepare

st.set_page_config(page_title="Customer Churn Predictor", layout="wide")


@st.cache_resource
def load_artifacts():
    model = joblib.load("models/best_churn_model.joblib")
    with open("models/model_metadata.json") as f:
        meta = json.load(f)
    return model, meta


model, meta = load_artifacts()

st.title("Customer Churn Predictor")
st.caption(f"Telco Customer Churn - model: {meta['best_model']} "
           f"(test ROC-AUC {meta['test_metrics']['ROC-AUC']:.3f})")
st.write("Enter the customer's details in the sidebar and press **Predict**.")

with st.sidebar:
    st.header("Customer details")
    gender = st.selectbox("Gender", ["Female", "Male"])
    senior = st.selectbox("Senior citizen", ["No", "Yes"])
    partner = st.selectbox("Has partner", ["No", "Yes"])
    dependents = st.selectbox("Has dependents", ["No", "Yes"])
    tenure = st.slider("Tenure (months)", 0, 72, 12)

    st.subheader("Services")
    phone = st.selectbox("Phone service", ["Yes", "No"])
    multi = st.selectbox("Multiple lines", ["No", "Yes"])
    internet = st.selectbox("Internet service", ["Fiber optic", "DSL", "No"])
    online_sec = st.selectbox("Online security", ["No", "Yes"])
    online_bkp = st.selectbox("Online backup", ["No", "Yes"])
    device = st.selectbox("Device protection", ["No", "Yes"])
    tech = st.selectbox("Tech support", ["No", "Yes"])
    stv = st.selectbox("Streaming TV", ["No", "Yes"])
    smov = st.selectbox("Streaming movies", ["No", "Yes"])

    st.subheader("Account")
    contract = st.selectbox("Contract", ["Month-to-month", "One year", "Two year"])
    paperless = st.selectbox("Paperless billing", ["Yes", "No"])
    payment = st.selectbox("Payment method", ["Electronic check", "Mailed check",
                                              "Bank transfer (automatic)",
                                              "Credit card (automatic)"])
    monthly = st.number_input("Monthly charges ($)", 0.0, 200.0, 70.0, step=0.5)
    total = st.number_input("Total charges ($)", 0.0, 10000.0, float(monthly * max(tenure, 1)), step=10.0)

if st.button("Predict", type="primary"):
    customer = {
        "gender": gender, "SeniorCitizen": 1 if senior == "Yes" else 0, "Partner": partner,
        "Dependents": dependents, "tenure": tenure, "PhoneService": phone, "MultipleLines": multi,
        "InternetService": internet, "OnlineSecurity": online_sec, "OnlineBackup": online_bkp,
        "DeviceProtection": device, "TechSupport": tech, "StreamingTV": stv, "StreamingMovies": smov,
        "Contract": contract, "PaperlessBilling": paperless, "PaymentMethod": payment,
        "MonthlyCharges": monthly, "TotalCharges": total,
    }
    feats = prepare(pd.DataFrame([customer]), feature_columns=meta["all_feature_columns"])
    prob = float(model.predict_proba(feats[meta["selected_features"]])[0, 1])

    c1, c2 = st.columns(2)
    c1.metric("Churn probability", f"{prob:.1%}")
    if prob >= 0.5:
        c2.error("High risk - this customer is likely to churn. Consider a retention offer.")
    elif prob >= 0.3:
        c2.warning("Medium risk - worth monitoring.")
    else:
        c2.success("Low risk - this customer is likely to stay.")
    st.progress(min(max(prob, 0.0), 1.0))

In [ ]:
%%writefile requirements.txt
streamlit
scikit-learn
pandas
numpy
joblib
xgboost

## Download the project files
This packs the model, helper module, app and results tables into one zip so you can add them to your GitHub repository.

In [ ]:
final_df.round(4).to_csv("model_results_test.csv")
cv_table.round(4).to_csv("model_results_cv.csv")

import shutil
os.makedirs("project_export", exist_ok=True)
for f in ["churn_utils.py", "app.py", "requirements.txt", "model_results_test.csv", "model_results_cv.csv"]:
    shutil.copy(f, "project_export")
shutil.copytree("models", "project_export/models", dirs_exist_ok=True)
shutil.make_archive("Task_03_Churn_Project", "zip", "project_export")

try:
    from google.colab import files
    files.download("Task_03_Churn_Project.zip")
except ImportError:
    print("Created Task_03_Churn_Project.zip (not running in Colab, so no auto-download).")